# Browser Automation & Web Agents

---
> **SECRETS SETUP:** Click the 🔑 key icon in the left sidebar → Add secret:
> - **Name:** `GROQ_API_KEY`
> - **Value:** `gsk_...` (your key from https://console.groq.com/keys)
> - Enable **"Notebook access"** toggle

In [ ]:
# ===================================================================
# ONE-TIME SETUP CELL (run first, takes 2-4 minutes in Colab)
# ===================================================================

# 1. Python packages.
#    browser-use 0.1.40 requires playwright>=1.49 and pydantic>=2.10.4,
#    so we pin a COMPATIBLE playwright and let pip resolve pydantic.
#    (The old pins playwright==1.47.0 / pydantic==2.9.2 caused
#     "ResolutionImpossible" and nothing installed.)
#    langchain==0.3.18 is pinned so Colab's stale pre-installed langchain
#    matches langchain-core 0.3.x (otherwise ChatGroq -> 'module langchain
#    has no attribute verbose'). RESTART THE RUNTIME after this cell.
!pip -q install browser-use==0.1.40 langchain-groq==0.2.0 langchain==0.3.18 playwright==1.49.0 nest_asyncio==1.6.0

# 2. Chromium browser binary + OS-level libraries
!playwright install chromium
!playwright install-deps chromium > /dev/null 2>&1

# 3. Allow nested event loops (Colab already has one running)
import nest_asyncio
nest_asyncio.apply()

print("Setup complete. Playwright, browser-use, langchain-groq installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.1/613.1 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.9/296.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [ ]:
# ===================================================================
# Load GROQ_API_KEY from Colab Secrets
# ===================================================================
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("GROQ_API_KEY loaded from Colab Secrets.")
except Exception as e:
    print("GROQ_API_KEY not found in Colab Secrets.")
    print("Add it via the key icon in the left sidebar, then re-run this cell.")
    print(f"Underlying error: {e}")

GROQ_API_KEY loaded from Colab Secrets.


---
## Section 1: Opening — Why Agents Need Browsers (0:00–0:15)

### The Last-Mile Problem

Most real-world services humans use every day — IRCTC, bank portals, state government sites, internal HR tools — **do not expose a usable public API**. They only offer a browser.

> **Analogy:** An API is like the Indian Railways — structured, scheduled, predictable. A browser is the city streets of Indiranagar. Messy, unpredictable, but that's where the real work happens.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/205/477/original/img1.jpeg?1782201346)

### Where this is used in industry

| Product | What it does |
|---|---|
| **OpenAI Operator** (Jan 2025) | LLM drives a browser to book flights, fill forms |
| **Anthropic Claude Computer Use** (late 2024) | Controls the full desktop; browser is the headline case |
| **Google Project Mariner** | Research agent living inside Chrome |
| **Perplexity Deep Research** | Actually visits pages, not just search APIs |
| **MakeMyTrip / Skyscanner** | Browser-based scraping of airline sites |

### Three buckets where agents need browsers
1. **Research** — read pages, follow links, summarize, cite
2. **Form filling** — visa applications, IRCTC, college admissions
3. **Data extraction** — price tracking, lead generation, market research

### MCQ 1
> **Which of the following is the strongest reason an AI agent would need a browser instead of just calling an API?**
>
> A) Browsers are faster than APIs for most tasks  
> B) Most real-world services humans use do not expose a usable public API  
> C) Browsers give the LLM access to more compute  
> D) APIs cannot return HTML
>
> ✅ **Answer: B**

---
## Section 2: Playwright Basics for Agent Integration (0:15–0:45)

### What is Playwright?

Playwright is a browser automation library by Microsoft (2020). Controls Chromium, Firefox, and WebKit from Python with a single API.

> **Analogy:** Selenium is the Hindustan Ambassador — legendary, but showing its age. Playwright is a modern EV: faster, smarter, built for how the web works today.

**Why Playwright wins for agents:**
1. **Auto-waiting** — waits for elements to be actionable before acting
2. **Network interception** out of the box
3. **Multiple browser contexts** in one process — cheap parallelism

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/205/540/original/img2.png?1782202839)

### Core Primitives

```
Browser            (the Chromium process)
  └── Context      (like an incognito window — own cookies/storage)
        └── Page   (a single tab)
```

**For agents:** 1 Browser → N Contexts (one per user) → pages per task. Cheap and correct.

In [ ]:
# Conceptual diagram — Browser → Context → Page hierarchy
print("Browser -> Context -> Page. Memorize this hierarchy.")
print()
print("Browser   = the entire Chrome process (expensive to start, start once)")
print("Context   = one incognito window (own cookies, storage, identity)")
print("Page      = one tab inside a context")

Browser -> Context -> Page. Memorize this hierarchy.

Browser   = the entire Chrome process (expensive to start, start once)
Context   = one incognito window (own cookies, storage, identity)
Page      = one tab inside a context


In [ ]:
# Simplest Playwright example - async API (Colab-friendly)
# Browser binaries were already installed in the setup cell above.
from playwright.async_api import async_playwright

async def hello_playwright():
    async with async_playwright() as p:
        # headless=True is mandatory on Colab (no display available)
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page    = await context.new_page()

        await page.goto("https://en.wikipedia.org/wiki/Transformer_(deep_learning)")
        title   = await page.title()
        heading = await page.locator("h1").inner_text()

        await browser.close()
        return title, heading

title, heading = await hello_playwright()
print(f"Page title: {title}")
print(f"H1 text:    {heading}")

Page title: Transformer (deep learning) - Wikipedia
H1 text:    Transformer (deep learning)


### Locators — How the Agent Points at Things

Priority order (most robust → most fragile):

| Strategy | Example | Stability |
|---|---|---|
| **Role-based** | `get_by_role("button", name="Add to cart")` | ✅ Best |
| **Text/Label** | `get_by_text("Forgot password?")` | ✅ Good |
| **data-testid** | `locator("[data-testid='price']")` | ✅ Good |
| **CSS class** | `locator("button.css-1xyz9q")` | ⚠️ Fragile |
| **XPath** | `locator("xpath=/html/body/div[3]/...")` | ❌ Avoid |

> **Rule for agents:** prefer role and text locators — they match how a human (and an LLM reading the accessibility tree) describes the page.

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/205/554/original/img3.png?1782203085)

In [ ]:
# Locator strategy examples (conceptual — not executed against a live page)

# ✅ Role-based (preferred)
# page.get_by_role("button", name="Add to cart")
# page.get_by_role("link",   name="Sign in")
# page.get_by_role("textbox", name="Email")

# ✅ Text / label
# page.get_by_text("Forgot password?")
# page.get_by_label("Email address")
# page.get_by_placeholder("Enter OTP")

# ⚠️ CSS / XPath (fallback only)
# page.locator("css=button.add-to-cart")
# page.locator("xpath=//div[@data-testid='price']")
# page.locator("[data-testid='price']")   # data-testid is the sweet spot

print("Locator priority: role > text/label > data-testid > CSS > XPath")

Locator priority: role > text/label > data-testid > CSS > XPath


In [ ]:
# Realistic Playwright demo: search Wikipedia.
# Why Wikipedia? Google Scholar / Google / DuckDuckGo all serve anti-bot
# challenge pages to Colab's datacenter IPs. Wikipedia is automation-friendly.
#
# We navigate straight to the full-text search RESULTS URL instead of typing
# into the live search box. Reason: Wikipedia's Vector-2022 skin hydrates the
# search box into a JavaScript typeahead widget and replaces the <input> node
# mid-interaction, so fill()+press("Enter") hits a "detached from the DOM"
# race and times out. Going to the underlying URL is the robust pattern when
# a JS widget is flaky.
from urllib.parse import quote_plus
from playwright.async_api import async_playwright

async def search_demo(query: str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/126.0.0.0 Safari/537.36"
            )
        )
        page = await context.new_page()

        # fulltext=1 forces the results-list page (even when an exact-title
        # article exists), so the parsing below is deterministic.
        await page.goto(
            f"https://en.wikipedia.org/w/index.php?fulltext=1&search={quote_plus(query)}&ns0=1",
            wait_until="domcontentloaded",
        )

        results = page.locator("ul.mw-search-results .mw-search-result-heading a")
        try:
            await results.first.wait_for(timeout=10000)
            titles = await results.all_inner_texts()
            kind = "search results"
        except Exception:
            # Fallback: if no list rendered, read the page heading.
            titles = [await page.locator("h1#firstHeading").inner_text()]
            kind = "direct article"

        await browser.close()
        return kind, titles[:3]

kind, top_results = await search_demo("browser automation software")
print(f"Outcome: {kind}")
for i, r in enumerate(top_results, 1):
    print(f"{i}. {r}")

Outcome: search results
1. Headless browser
2. Test automation
3. Selenium (software)


### Auto-Wait — Why Playwright is Agent-Friendly

When you call `page.click(...)`, Playwright first checks:
1. Element is **attached** to the DOM
2. Element is **visible**
3. Element is **stable** (not animating)
4. Element **receives events** (not covered by an overlay)
5. Element is **enabled**

Only when all five pass does the click fire — retrying up to the timeout (default 30s).

> For an LLM agent, this is huge. The model doesn't need to reason about page load timing.

**Explicit waits (when you need more control):**
```python
await page.wait_for_selector(".results", state="visible")  # ✅ preferred
await page.wait_for_url("**/dashboard")
await page.wait_for_load_state("networkidle")               # ⚠️ often wrong on modern sites
```

> ⚠️ `networkidle` is overused. Sites like Flipkart/Amazon never truly go idle due to background pings. Prefer waiting on a **specific element or URL pattern**.

### MCQ 2
> **You are building an agent that clicks "Add to Cart" on Flipkart. Which locator strategy is most robust against frontend redesigns?**
>
> A) `page.locator("button.css-1xyz9q")`  
> B) `page.locator("xpath=/html/body/div[3]/div/div[2]/button")`  
> C) `page.get_by_role("button", name="Add to cart")`  
> D) `page.locator("button:nth-child(7)")`
>
> ✅ **Answer: C** — Role + visible name matches the accessibility tree and survives redesigns. A, B, and D break the moment the DOM is restructured.

---
## Section 3: Dynamic Content, JavaScript Rendering & Authentication (0:45–1:10)

### Dynamic Content and SPAs

Most modern Indian consumer sites (Flipkart, Swiggy, Zomato, Cred) are **Single Page Applications**. They load a near-empty HTML shell, then JavaScript builds the page. `requests.get(url).text` returns almost nothing.

Playwright runs a real browser, so JavaScript executes normally. But you now need to know **when the page is "done"**.

**Three patterns:**
1. Wait for a specific element you actually need ✅ **(gold standard)**
2. Intercept the underlying network/API call for clean JSON
3. Infinite scroll — scroll, wait for new items, repeat (always with a cap!)

In [ ]:
# Infinite scroll demo — quotes.toscrape.com/scroll (safe practice site)
from playwright.async_api import async_playwright

async def infinite_scroll_demo():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page    = await browser.new_page()

        await page.goto("https://quotes.toscrape.com/scroll", wait_until="domcontentloaded")
        await page.wait_for_selector(".quote", timeout=10000)

        previous_count = 0
        for _ in range(8):          # ← ALWAYS set a safety cap
            await page.mouse.wheel(0, 5000)
            await page.wait_for_timeout(1200)
            current_count = await page.locator(".quote").count()
            if current_count == previous_count:
                break
            previous_count = current_count

        total = await page.locator(".quote").count()
        await browser.close()
        return total

total_quotes = await infinite_scroll_demo()
print(f"Loaded {total_quotes} quotes via infinite scroll.")

Loaded 80 quotes via infinite scroll.


### Authentication Pattern 1: `storage_state` (Production Pattern)

**The problem:** Logging in from scratch every run is slow, triggers anti-bot heuristics, and may trigger 2FA.

**The solution:** Log in once → save cookies + localStorage to a JSON file → every subsequent run loads that file.

> **Analogy:** The "Remember me on this device" checkbox. You authenticate once; your machine becomes a trusted device. `storage_state.json` is the agent's trusted-device certificate.

**On Colab (no interactive display), the flow is:**
1. Run an interactive script **locally** (`headless=False`), log in by hand, save the state file
2. Upload that file to Colab
3. Load the state file in Colab

> ⚠️ **Security:** `auth_state.json` contains session cookies. Anyone with this file can act as you. Treat it like an SSH key — never commit it, add to `.gitignore`, store encrypted in production (AWS Secrets Manager, Vault, etc.)

In [ ]:
# STEP 1 — Run this LOCALLY (not in Colab): one-time manual login
# Copy-paste this into a local Python file and run it on your laptop.

LOCAL_SCRIPT = '''
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)   # headed so you can see & type
    context = browser.new_context()
    page    = context.new_page()
    page.goto("https://github.com/login")
    input("Log in manually in the browser, then press Enter here...")
    context.storage_state(path="github_state.json")
    browser.close()
    print("Session saved to github_state.json")
'''
print(LOCAL_SCRIPT)
print("After running locally, upload github_state.json via Colab's file pane (📁 icon).")


from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)   # headed so you can see & type
    context = browser.new_context()
    page    = context.new_page()
    page.goto("https://github.com/login")
    input("Log in manually in the browser, then press Enter here...")
    context.storage_state(path="github_state.json")
    browser.close()
    print("Session saved to github_state.json")

After running locally, upload github_state.json via Colab's file pane (📁 icon).


In [ ]:
# STEP 2 — Reuse the saved session in Colab
import os
from playwright.async_api import async_playwright

async def run_with_saved_session(target_url: str, state_path: str = "github_state.json"):
    if not os.path.exists(state_path):
        print(f"No session file at {state_path}.")
        print("Upload it via Colab's file pane (📁 icon) first.")
        return None

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        # ↓ pass storage_state here — starts already logged in
        context = await browser.new_context(storage_state=state_path)
        page    = await context.new_page()
        await page.goto(target_url)
        title = await page.title()
        await browser.close()
        return title

title = await run_with_saved_session("https://github.com/settings/profile")
print(f"Result: {title}")

No session file at github_state.json.
Upload it via Colab's file pane (📁 icon) first.
Result: None


### Authentication Pattern 2: Programmatic Login

Use when: rotating accounts, service accounts with credentials in a secret manager, or fresh onboarding flows.

> ⚠️ Demo below uses **publicly published credentials** from a practice site (`the-internet.herokuapp.com`). In real code, pull from Colab Secrets — never hardcode passwords.

In [ ]:
# Programmatic login demo — practice site with published test credentials
# Username: tomsmith | Password: SuperSecretPassword!
from playwright.async_api import async_playwright

async def programmatic_login_demo():
    # In production: pull from Colab Secrets
    # username = userdata.get("MY_SERVICE_USERNAME")
    username = "tomsmith"
    password = "SuperSecretPassword!"

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page    = await context.new_page()

        await page.goto("https://the-internet.herokuapp.com/login")

        await page.get_by_label("Username").fill(username)
        await page.get_by_label("Password").fill(password)
        await page.get_by_role("button", name="Login").click()

        await page.wait_for_url("**/secure", timeout=10000)
        flash_message = (await page.locator("#flash").inner_text()).strip()

        # Save session for reuse — combine both patterns!
        await context.storage_state(path="practice_state.json")

        await browser.close()
        return flash_message

msg = await programmatic_login_demo()
print(f"Login result: {msg[:120]}...")
print("Session saved to practice_state.json (visible in Colab file pane 📁).")

Login result: You logged into a secure area!
×...
Session saved to practice_state.json (visible in Colab file pane 📁).


### MCQ 3
> **Your agent needs to interact with a logged-in dashboard on a SaaS tool every hour. Which approach is best?**
>
> A) Log in fresh with username and password every hour  
> B) Log in once manually, save `storage_state`, reuse it, and add re-auth logic when the session expires  
> C) Disable the site's login by editing cookies  
> D) Use the same browser window indefinitely and never close it
>
> ✅ **Answer: B** — Option A triggers anti-bot alarms. Option C is nonsense. Option D fails on process restart. `storage_state` is the production-grade answer.

---
## ☕ Break (1:10–1:15)

Stretch, hydrate. We pick up with **browser-use** — the open-source equivalent of what powers OpenAI Operator and Claude Computer Use.

In [ ]:
print("Break time. Back in 5 minutes.")

Break time. Back in 5 minutes.


---
## Section 5: browser-use Library Deep Dive (1:15–1:45)

### What is browser-use and why does it exist?

Playwright gives you eyes and hands. But there's a gap between `page.click('button.add-to-cart')` and "my LLM decides on its own what to click on a page it has never seen."

**browser-use** fills that gap. It's an open-source Python library that sits on top of Playwright and gives your LLM a structured way to perceive and act on web pages.

> **Analogy:** Playwright is the steering wheel and pedals. browser-use is the chauffeur who can read road signs. You give instructions in natural language; the chauffeur translates them into precise mechanical actions.

**What browser-use does for you:**
1. **Perception** — serializes the page into a representation the LLM can reason about
2. **Action space** — fixed vocabulary: click element 7, type into element 12, scroll, go back
3. **Loop** — perceive → think → act, including retries and self-correction
4. **Extensibility** — custom actions, any LangChain-compatible LLM, Pydantic output

### How the LLM "sees" a page

**Mode 1: DOM serialization with element indexing (default)**  
browser-use walks the DOM, finds all interactive elements, filters invisible ones, and produces a numbered list:
```
[1]<button>Sign in</button>
[2]<input type="email" placeholder="Email"/>
[3]<input type="password" placeholder="Password"/>
[4]<a>Forgot password?</a>
[5]<button>Continue with Google</button>
```
The LLM sees this, decides "click element 1", emits `click_element(index=1)`. Fast, token-efficient, robust.

**Mode 2: Vision (screenshot)**  
browser-use takes a screenshot, draws colored bounding boxes with index numbers, and sends it to a vision-capable LLM. More reliable for visually complex pages (Canva, Figma) but costs more tokens.

> For text-only Groq models like `llama-3.3-70b-versatile`, we use **DOM-only mode** (`use_vision=False`).

In [ ]:
# Minimal browser-use agent driven by Groq
import os

async def run_basic_agent():
    from browser_use import Agent, Browser, BrowserConfig
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        print("GROQ_API_KEY not loaded. Add it in Colab Secrets and re-run the secrets cell.")
        return None

    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0,
    )

    # On Colab there is NO display, so the browser MUST be headless.
    # browser-use defaults BrowserConfig(headless=False), which crashes here,
    # so we pass an explicit headless browser.
    browser = Browser(config=BrowserConfig(headless=True))

    agent = Agent(
        task=(
            "Go to https://news.ycombinator.com, find the top story, "
            "and return its title and the number of points."
        ),
        llm=llm,
        browser=browser,
        use_vision=False,       # llama-3.3 is text-only
    )

    try:
        history = await agent.run(max_steps=15)   # <- ALWAYS set max_steps
        return history.final_result()
    finally:
        await browser.close()

result = await run_basic_agent()
print("Final result:")
print(result)

/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  value['message'] = load(value['message'])
/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  value['message'] = load(value['message'])
/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  value['message'] = load(value['message'])
/usr/local

Final result:
The title of the top story is Unlimited OCR: One-Shot Long-Horizon Parsing and it has 93 points.


### Intermediate: Structured Output with Pydantic

Instead of free-form text, get a typed Python object. This is the pattern you'll use 90% of the time in production — the LLM is forced into a known schema.

In [ ]:
# Structured output - the production pattern, Groq-powered
import os
from typing import List
from pydantic import BaseModel

class HNStory(BaseModel):
    title:    str
    points:   int
    url:      str
    comments: int

class HNTop(BaseModel):
    stories: List[HNStory]

async def run_structured_agent():
    from browser_use import Agent, Controller, Browser, BrowserConfig
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        print("GROQ_API_KEY not loaded.")
        return None

    controller = Controller(output_model=HNTop)
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    browser = Browser(config=BrowserConfig(headless=True))   # headless on Colab

    agent = Agent(
        task=(
            "Go to https://news.ycombinator.com and extract the top 5 stories. "
            "For each story, capture the title, points, URL, and comment count."
        ),
        llm=llm,
        controller=controller,
        browser=browser,
        use_vision=False,
    )

    try:
        history = await agent.run(max_steps=20)
        final = history.final_result()
        if final:
            return HNTop.model_validate_json(final)
        return None
    finally:
        await browser.close()

parsed = await run_structured_agent()
if parsed:
    for s in parsed.stories:
        print(f"- {s.title} | {s.points} pts | {s.comments} comments")

/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  value['message'] = load(value['message'])
/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  value['message'] = load(value['message'])
/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages'

- Unlimited OCR: One-Shot Long-Horizon Parsing | 93 pts | 26 comments
- The Coming Loop | 72 pts | 42 comments
- Steam Machine launches today | 1716 pts | 1459 comments
- Plotnine | 105 pts | 25 comments
- Will It Mythos? | 199 pts | 133 comments


### Custom Actions

When the default action vocabulary isn't enough, register your own. Production agents have entire libraries of domain-specific actions.

In [ ]:
# Custom action — register a tool the agent can choose to call
from browser_use import Controller, ActionResult

custom_controller = Controller()

@custom_controller.action("Save an important note for the user to read later")
def save_note(note: str) -> ActionResult:
    # In production: write to a database, send to Slack, etc.
    with open("agent_notes.txt", "a") as f:
        f.write(note + "\n")
    return ActionResult(extracted_content=f"Note saved: {note}")

print("Custom action 'save_note' registered on custom_controller.")
print("Pass controller=custom_controller to Agent(...) to make it available.")

Custom action 'save_note' registered on custom_controller.
Pass controller=custom_controller to Agent(...) to make it available.


### Controlling the Loop: System Prompts and Policies

The `message_context` parameter is where you encode your agent's **policies** — the equivalent of writing an SOP for your chauffeur. (In browser-use 0.1.40 this is the supported hook for injecting extra instructions; later versions add `extend_system_message`.)

In [ ]:
# Tuned agent with explicit policies and custom browser config
import os

async def run_tuned_agent():
    from browser_use import Agent, Browser, BrowserConfig
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        print("GROQ_API_KEY not loaded.")
        return None

    browser = Browser(
        config=BrowserConfig(
            headless=True,           # always headless on Colab
            disable_security=False,  # leave site security ON
        )
    )

    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

    extra_instructions = (
        "You are a careful research assistant. "
        "Never click on advertisements. "
        "If a page asks for login, stop and report 'auth required'. "
        "Cite the URL of any fact you report."
    )

    agent = Agent(
        task="Find the current top headline on https://www.bbc.com/news and report it with its URL.",
        llm=llm,
        browser=browser,
        message_context=extra_instructions,   # 0.1.40 hook for extra policy text
        use_vision=False,
    )

    try:
        history = await agent.run(max_steps=10)
        return history.final_result()
    finally:
        await browser.close()

tuned_result = await run_tuned_agent()
print("Tuned agent result:")
print(tuned_result)

/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  value['message'] = load(value['message'])
/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  value['message'] = load(value['message'])
/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages'

Tuned agent result:
The current top headline on https://www.bbc.com/news is 'Hundreds of schools partially closed as UK braces for hottest June day on record' with URL https://www.bbc.com/news/live/c0ryvlxpg7rt


### When to Use browser-use vs Raw Playwright

| Situation | Use |
|---|---|
| Same site, same flow, every time (e.g., daily report download) | **Raw Playwright** |
| Unknown sites, exploratory tasks, natural-language goals | **browser-use** |
| Critical, must-not-fail business process | **Raw Playwright** (browser-use as fallback) |
| Research, "find me something," summarization | **browser-use** |
| High volume, tight latency, cost-sensitive | **Raw Playwright** (LLM calls add latency + cost) |
| Building products like Operator or Mariner | **browser-use** or custom equivalent |

> **The professional pattern is hybrid:** Use raw Playwright for the deterministic skeleton (login, navigate, click Export). Use browser-use only for the squishy, judgment-requiring middle step.

### MCQ 4
> **Your agent must extract product prices from 50 different competitor websites every morning. Some you've scraped before; many are new. Best architecture?**
>
> A) Pure raw Playwright with hand-written selectors for all 50 sites  
> B) Pure browser-use for everything  
> C) Raw Playwright for the 10 stable sites you scrape often, browser-use for new/changing ones, with Pydantic structured output  
> D) Hit each site's homepage with `requests.get` and parse HTML
>
> ✅ **Answer: C** — Deterministic and cheap where you can, LLM-driven where you must, structured output everywhere. Option A doesn't scale (50 selector sets to maintain). Option B is slow and expensive at scale. Option D fails on every SPA.

---
## Section 6: Safety, Rate Limiting & Respectful Scraping (1:45–1:55)

### The Honest Framing on Legality and Ethics

Web scraping occupies a legal grey zone that varies by country, site, and use case.

**Three layers to understand:**
1. **`robots.txt`** — voluntary file at the root of every site. Not legally binding in most jurisdictions, but ignoring it is a clear signal of bad faith.
2. **Terms of Service** — most sites prohibit automated access. The *hiQ v LinkedIn* case (2022 US) established that scraping **public** data is generally not a CFAA violation, but login-gated content is messier.
3. **Data protection laws** — GDPR (EU), DPDP Act (India, 2023), CCPA (California). Personal data = regulated territory.

> **The lawyer test:** If you'd be embarrassed to explain your scraper to the site's lawyer, don't run it. This single test catches 95% of bad decisions.

**Three things you should always do:**
1. **Identify yourself** — set a user-agent with a contact email: `MyBot/1.0 (contact: you@example.com)`
2. **Respect `robots.txt`** unless you have explicit permission
3. **Avoid login-gated content** unless you are the account owner

In [ ]:
# Respectful scraping helpers — async, Colab-friendly
import asyncio
import random

async def polite_delay(min_seconds: float = 1.0, max_seconds: float = 3.0):
    """Sleep a random amount between requests. Random jitter looks less robotic."""
    delay = random.uniform(min_seconds, max_seconds)
    await asyncio.sleep(delay)

async def with_exponential_backoff(coro_factory, max_attempts: int = 5):
    """
    Retry an async operation with exponential backoff + jitter.
    coro_factory: a callable that returns a fresh coroutine each call.
    """
    for attempt in range(max_attempts):
        try:
            return await coro_factory()
        except Exception as e:
            if attempt == max_attempts - 1:
                raise
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"Attempt {attempt + 1} failed ({e}). Waiting {wait:.1f}s.")
            await asyncio.sleep(wait)

class ConcurrencyLimiter:
    """Never more than N pages in flight at once."""
    def __init__(self, max_concurrent: int = 3):
        self.semaphore = asyncio.Semaphore(max_concurrent)

    async def run(self, coro):
        async with self.semaphore:
            return await coro

# Sanity check
async def _demo():
    await polite_delay(0.2, 0.5)
    return "polite delay complete"

print(await _demo())
print("Helpers ready: polite_delay, with_exponential_backoff, ConcurrencyLimiter")

polite delay complete
Helpers ready: polite_delay, with_exponential_backoff, ConcurrencyLimiter


### Rate Limiting Rules of Thumb

1. **Default to 1–3 concurrent requests per site** (not per machine — per site)
2. **Add random jitter** — fixed `time.sleep(2)` is a fingerprint; `random.uniform(1.0, 3.5)` looks human
3. **Honor `429 Too Many Requests` and `Retry-After` headers** — if the site tells you to slow down, slow down

> **Analogy:** Don't be the delivery rider who runs every red light to save 30 seconds. You'll eventually get caught, banned, or cause harm.

---
## Section 7: Debugging Browser Automation Failures (1:55–2:05)

### The Top 5 Failure Modes

| # | Failure | Fix |
|---|---|---|
| 1 | **Selector changed** — frontend redesign overnight | Use role/text locators; monitor for selector misses |
| 2 | **Timing/race condition** — element not yet visible | Trust auto-wait; use `wait_for_selector(state="visible")` |
| 3 | **Auth expired** — redirected to `/login` | Detect login redirects explicitly; refresh session proactively |
| 4 | **Anti-bot detection** — Cloudflare/Datadome challenge page | Realistic user-agent, slow down, reconsider if you should automate this |
| 5 | **Agent loop** — LLM gets confused and repeats actions | Hard `max_steps` cap; strong system prompts; good logging |

### The Three Debugging Tools (in order)
1. **Headed mode** (`headless=False`, on your laptop) — watch the browser. Solves more bugs than reading logs.
2. **Playwright Trace Viewer** — records every action, DOM snapshot, and network call. Replay like a video.
3. **browser-use history** — full action trace, LLM messages, and final result.

In [ ]:
# Recording a Playwright trace for later inspection
from playwright.async_api import async_playwright

async def run_with_trace():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        await context.tracing.start(screenshots=True, snapshots=True, sources=True)

        page = await context.new_page()
        await page.goto("https://example.com")
        await page.locator("h1").inner_text()

        await context.tracing.stop(path="trace.zip")
        await browser.close()

    print("Trace written to trace.zip")
    print("Download it from the Colab file pane (📁), then on your laptop run:")
    print("  playwright show-trace trace.zip")

await run_with_trace()

Trace written to trace.zip
Download it from the Colab file pane (📁), then on your laptop run:
  playwright show-trace trace.zip


In [ ]:
# Inspecting browser-use agent history for debugging
import os

async def inspect_history():
    from browser_use import Agent, Browser, BrowserConfig
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        print("GROQ_API_KEY not loaded.")
        return None

    llm   = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    browser = Browser(config=BrowserConfig(headless=True))   # headless on Colab
    agent = Agent(
        task="Go to https://example.com and report the page title.",
        llm=llm,
        browser=browser,
        use_vision=False,
    )

    try:
        history = await agent.run(max_steps=5)

        print("=== Agent history summary ===")
        print(f"Total steps:       {len(history.history)}")
        print(f"Final result:      {history.final_result()}")
        print(f"URLs visited:      {history.urls()}")
        print(f"Errors:            {history.errors()}")

        return history
    finally:
        await browser.close()

_ = await inspect_history()

/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  value['message'] = load(value['message'])
/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  value['message'] = load(value['message'])
/usr/local/lib/python3.12/dist-packages/browser_use/agent/message_manager/views.py:59: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages'

=== Agent history summary ===
Total steps:       4
Final result:      The page title of https://example.com is: Example Domain
URLs visited:      ['about:blank', 'https://example.com/', 'https://example.com/', 'https://example.com/']
Errors:            [None, None, 'Error code: 400 - {\'error\': {\'message\': \'tool call validation failed: parameters for tool AgentOutput did not match schema: errors: [`/action/0/done/success`: expected boolean, but got string, `/action/0/done`: expected null, but got object]\', \'type\': \'invalid_request_error\', \'code\': \'tool_use_failed\', \'failed_generation\': \'<function=AgentOutput>{"current_state": {"evaluation_previous_goal": "Success - Extracted page title", "memory": "Visited https://example.com and extracted the page title. 1 out of 1 websites visited.", "next_goal": "Report the page title"}, "action": [{"done": {"success": "true", "text": "The page title of https://example.com is: Example Domain"}}]}</function>\'}}', None]


### MCQ 5
> **Your browser-use agent worked perfectly yesterday and fails today on the same task. Most likely cause?**
>
> A) The LLM provider is down  
> B) The site shipped a UI change, your session expired, or anti-bot kicked in  
> C) Playwright itself is broken  
> D) Python upgraded overnight
>
> ✅ **Answer: B** — "The site changed something" or "session expired" or "rate limited" accounts for the vast majority of overnight failures. Always check those three before suspecting your code.

---
## Recap & Homework

### What we covered

1. **Why agents need browsers** — the last-mile problem; APIs are railways, browsers are city streets
2. **Playwright basics** — Browser → Context → Page; role/text locators first; auto-wait is the magic
3. **Dynamic content** — SPAs need real browsers; wait on specific elements, not `networkidle`
4. **Authentication** — `storage_state` is the production pattern; treat session files like SSH keys
5. **browser-use** — DOM serialization + indexed elements + optional vision; drive with Groq; extend with Pydantic output and custom actions
6. **Safety** — respect `robots.txt`, identify yourself, rate-limit with jitter, honor `429`; use the lawyer test
7. **Debugging** — top 5 failure modes; Playwright traces; browser-use history